# AI6 Workshop 6W: Optimisation in the Wild (Model Tuning)

This lab is the “maths made practical” part of Unit 6.

Not because we’re doing calculus on a whiteboard — but because we’re going to use the *real* maths vocabulary that shows up in papers and tooling: **loss**, **gradients (partial derivatives)**, **stochastic gradient descent (SGD)**, and **generalisation**.

The engineering question we’re answering is:
**is training working, and what dial should we turn next?**

**Story:** A pretrained model is like hiring a graduate who already speaks excellent English. Fine‑tuning is onboarding them into finance so phrases like *“guidance cut”* trigger the right instinct. We are not teaching English again; we are adjusting behaviour.

In Unit 6.2 you learned that training is **optimisation**: we measure how wrong we are (**loss**), use the **gradient** (slope) to pick a direction, and take repeated steps that make the model less wrong. The **learning rate** controls step size.

In this notebook you will:
1. Run a **baseline** evaluation (pretrained model + random classification head)
2. Fine‑tune using one learning‑rate preset (often in groups)
3. Compare before/after metrics and plot a loss curve
4. Diagnose generalisation using the **generalisation gap** (train vs validation)
5. Explain what happened in plain English


In [ ]:
# If you need to install dependencies inside the notebook, run this cell.
# Otherwise, prefer: pip install -r requirements.txt

# !pip install -r ../requirements.txt


In [ ]:
import os
import numpy as np
import pandas as pd

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    set_seed,
)

from sklearn.metrics import accuracy_score, f1_score
import matplotlib.pyplot as plt

set_seed(42)


## Speed dials (edit if your machine is slow)

These settings deliberately trade quality for speed so the workshop runs on typical laptops.

- `TRAIN_SUBSET`: reduce to 300-500 if training is slow  
- `EPOCHS`: set to 1 for a fast run  
- `MAX_LENGTH`: reduce to 64 to speed up tokenisation/training


In [ ]:
# Speed dials (make it run on typical laptops)
TRAIN_SUBSET = 800   # set to 0 to use the full train split
EPOCHS = 2
MAX_LENGTH = 96

# Tuning dial (choose ONE config for your run)
LR_PRESET = "JUST_RIGHT"   # options: TOO_LOW, JUST_RIGHT, TOO_HIGH
SCHEDULER = "constant"     # options: constant, linear (optional)

# Optional: experiment tracking (only if policies allow)
USE_WANDB = False
WANDB_PROJECT = "ai6-6w-model-tuning"


## 1) Load dataset (Hugging Face with local fallback)

Primary path: `financial_phrasebank` (sentences_allagree)  
Fallback: `lab/data/synth_financial_sentiment.csv`

If you're offline or downloads are blocked, the fallback dataset still supports the learning objectives.


In [ ]:
from datasets import Dataset

def _resolve_synth_csv_path():
    """Find the local synthetic dataset regardless of where Jupyter was launched from."""
    candidates = [
        os.path.join("lab", "data", "synth_financial_sentiment.csv"),  # launched from repo root
        os.path.join("data", "synth_financial_sentiment.csv"),         # launched from lab/
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"Could not find synth dataset. Tried: {candidates}")

def load_financial_sentiment_dataset():
    try:
        ds = load_dataset("financial_phrasebank", "sentences_allagree")["train"]
        ds = ds.rename_column("sentence", "text").rename_column("label", "labels")
        ds_source = "huggingface:financial_phrasebank/sentences_allagree"
        label_names = ds.features["labels"].names  # e.g., ['negative', 'neutral', 'positive']
        return ds, ds_source, label_names
    except Exception:
        # Local fallback (offline-friendly)
        csv_path = _resolve_synth_csv_path()
        ds = load_dataset("csv", data_files=csv_path)["train"]
        ds = ds.rename_column("sentence", "text")
        # Map string labels -> ints
        label_order = ["negative", "neutral", "positive"]
        label2id = {name: i for i, name in enumerate(label_order)}
        ds = ds.map(lambda x: {"labels": label2id[x["label"]]})
        ds = ds.remove_columns(["label"])
        ds_source = f"local:{csv_path}"
        label_names = label_order
        return ds, ds_source, label_names

dataset, dataset_source, label_names = load_financial_sentiment_dataset()
dataset_source, len(dataset), label_names, dataset[0]


## 2) Train/validation split (stratified)

We want a fair validation set, so we split and (where possible) stratify by class.


In [ ]:
# Stratified split if possible
try:
    split = dataset.train_test_split(test_size=0.2, seed=42, stratify_by_column="labels")
except Exception:
    split = dataset.train_test_split(test_size=0.2, seed=42)

train_ds = split["train"]
val_ds = split["test"]

# Optional speed dial: train on a subset for workshop speed
if TRAIN_SUBSET and TRAIN_SUBSET > 0:
    train_ds = train_ds.shuffle(seed=42).select(range(min(TRAIN_SUBSET, len(train_ds))))

print("Train size:", len(train_ds))
print("Val size:", len(val_ds))


## 3) Tokenise

Tokenisation turns text into input IDs + attention masks (model-ready tensors).


In [ ]:
try:
    tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
except Exception as e:
    print("ERROR: Could not load tokenizer/model from Hugging Face.")
    print("If you're offline or downloads are blocked, use the backup notebook:")
    print("  lab/02_backup_sgd_text_classifier_learning_rate.ipynb")
    raise

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

train_tok = train_ds.map(tokenize, batched=True)
val_tok = val_ds.map(tokenize, batched=True)

cols = ["input_ids", "attention_mask", "labels"]
train_tok.set_format("torch", columns=cols)
val_tok.set_format("torch", columns=cols)


## 4) Metrics + baseline evaluation

Two quick truths that make this workshop easier:

1) **Loss is like altitude.** It tells you whether training is going downhill (becoming less wrong mathematically).

2) **Validation accuracy/F1 is like a road test.** It tells you whether the model behaves better on examples it hasn’t seen.

Accuracy is “% correct”. **Macro‑F1** is stricter: it rewards balanced performance across classes (each class counts equally).

The relationship between training and validation is the **generalisation gap**:
> Generalisation gap = train performance − validation performance.  
> If the gap widens, you’re likely memorising (overfitting), not generalising.

Hugging Face `Trainer` will not show accuracy/F1 unless we define `compute_metrics`.

**Baseline** here means: *pretrained language model + random classification head.*

So you should expect near‑chance performance at baseline. That’s not failure — it’s the starting line.


In [ ]:
id2label = {i: name for i, name in enumerate(label_names)}
label2id = {name: i for i, name in enumerate(label_names)}

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

def build_model():
    return AutoModelForSequenceClassification.from_pretrained(
        "distilbert-base-uncased",
        num_labels=len(label_names),
        id2label=id2label,
        label2id=label2id,
    )


In [ ]:
# Baseline (untrained head) evaluation
baseline_model = build_model()

baseline_args = TrainingArguments(
    output_dir="./_baseline",
    per_device_eval_batch_size=16,
    report_to="none",
)

baseline_trainer = Trainer(
    model=baseline_model,
    args=baseline_args,
    eval_dataset=val_tok,
    compute_metrics=compute_metrics,
)

baseline_metrics = baseline_trainer.evaluate()
baseline_metrics


## 5) Choose a learning rate preset

This is the main tuning dial for the workshop.

Typical transformer fine‑tuning learning rates are around **2e‑5 to 5e‑5**.

Here we give you three presets:

- **TOO_LOW**: barely learns in 1–2 epochs  
- **JUST_RIGHT**: steady improvement  
- **TOO_HIGH**: intentionally aggressive (may oscillate, explode, or even error)

If you're working in groups: each group picks a different preset, then you compare curves and metrics.


In [ ]:
LR_MAP = {
    "TOO_LOW": 1e-6,
    "JUST_RIGHT": 2e-5,
    # Intentionally aggressive. Depending on hardware + randomness this may diverge or oscillate.
    "TOO_HIGH": 5e-3,
}

learning_rate = LR_MAP[LR_PRESET]
print("Preset:", LR_PRESET, "-> learning_rate =", learning_rate)


## 6) Train (fine‑tune)

This is where the optimisation loop actually runs.

When you call `trainer.train()` the model repeats this:

1) read a mini‑batch of labelled examples  
2) predict labels  
3) measure wrongness (**loss**)  
4) compute **gradients** (partial derivatives of the loss w.r.t. the weights)  
5) apply a small update using an optimiser (for transformers, often **AdamW**, in the SGD family)  
6) repeat

Important: this does **not** wipe the model’s previous knowledge. It nudges weights gradually. If you train too aggressively (very high LR, many epochs on narrow data) a model can become overly specialised — one reason we watch validation signals.

We log loss each epoch so we can plot the curve.


In [ ]:
model = build_model()

# Decide where to report logs (default: none)
report_to = "none"
run_name = None
if USE_WANDB:
    # Optional: Weights & Biases tracking (only if allowed by your organisation/policies)
    import wandb
    report_to = "wandb"
    run_name = f"{LR_PRESET}-{SCHEDULER}-e{EPOCHS}-n{TRAIN_SUBSET or 'all'}"
    wandb.init(project=WANDB_PROJECT, name=run_name)

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="no",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    learning_rate=learning_rate,
    lr_scheduler_type=SCHEDULER,   # constant or linear
    warmup_ratio=0.1 if SCHEDULER != "constant" else 0.0,
    weight_decay=0.01,
    seed=42,
    report_to=report_to,
    run_name=run_name,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    compute_metrics=compute_metrics,
)

TRAINING_FAILED = False
try:
    train_output = trainer.train()
    train_output
except Exception as e:
    TRAINING_FAILED = True
    print("⚠️ Training failed (this can happen with an aggressively high learning rate).")
    print("Error:", repr(e))
    print("Try lowering the learning rate preset, or reduce EPOCHS / TRAIN_SUBSET.")


## 6b) Did your training converge?

**Convergence** means that further training steps are producing negligible improvement —
the model has reached a point where the loss is no longer decreasing in any meaningful way.

Detecting convergence matters for two practical reasons:

**1. Reliability.** A model that has not converged is still moving. Its weights are not
yet stable, and the performance metrics you are reading at that point may not reflect
how the model would behave in deployment.

**2. Cost.** Compute is not free. Training beyond convergence wastes time, energy, and
money without producing a better model. In a business context, that cost compounds quickly
across repeated experiments — and it carries a real carbon footprint.

A simple convergence check: compare the loss change between your final two logged epochs.
If the change is smaller than a threshold you consider negligible (e.g. < 0.01 for this
task), your run has likely converged within the time budget.

> **Reflect:** Did your run converge? If you stopped training now, would you be confident
> that the model has stabilised — or are you stopping for practical reasons (time, cost)
> rather than because the curve has flattened? Record your answer in the convergence
> check section of your worksheet (Part B).
>
> **K18 link:** The relationship between the convergence condition and compute cost is
> exactly what your K18 Task 6 reflection asks about. A disciplined engineer does not run
> training until time runs out — they watch for convergence and stop when the curve
> tells them to.

In [ ]:
# Convergence check
# Reads from the training log produced by trainer.train() above.

if TRAINING_FAILED:
    print("Skipping convergence check — training did not complete.")
else:
    _log = trainer.state.log_history
    _train_losses = [
        row["loss"]
        for row in _log
        if "loss" in row and "eval_loss" not in row
    ]

    if len(_train_losses) >= 2:
        final_delta = abs(_train_losses[-1] - _train_losses[-2])
        converged = final_delta < 0.01

        print("=" * 50)
        print("CONVERGENCE CHECK")
        print("=" * 50)
        print(f"  Epoch {len(_train_losses) - 1} loss : {_train_losses[-2]:.4f}")
        print(f"  Epoch {len(_train_losses)} loss     : {_train_losses[-1]:.4f}")
        print(f"  Delta (change)              : {final_delta:.4f}")
        print()

        if converged:
            print("  \u2713  This run APPEARS TO HAVE CONVERGED within the time budget.")
            print("     Further epochs are unlikely to produce significant improvement.")
            print("     Stopping here is a defensible engineering decision.")
        else:
            print("  \u26a0  This run has NOT clearly converged.")
            print("     Training loss is still changing meaningfully.")
            print("     Practical question: is the remaining improvement worth")
            print("     the additional compute cost and time for this business goal?")

        print()
        print("Business framing:")
        print(f"  You ran {EPOCHS} epoch(s) on "
              f"{TRAIN_SUBSET if TRAIN_SUBSET else 'all'} training examples.")
        print("  Consider: how would you justify this compute spend to a project manager?")
        print("  Would a shorter run have been 'good enough' for the goal?")
        print()
        print("Sustainability note:")
        print("  Every training epoch consumes energy. The threshold you choose for")
        print("  'good enough' convergence is also an environmental and cost decision,")
        print("  not just a mathematical one.")

    elif len(_train_losses) == 1:
        print("Only one epoch was logged — cannot compute a delta.")
        print("Consider running at least 2 epochs to assess convergence.")
    else:
        print("No training loss values found in log history.")
        print("Check that trainer.train() completed and logging_strategy='epoch' is set.")

## 7) Evaluate after training + compare with baseline

In [ ]:
# Evaluate after training (if training succeeded)
if TRAINING_FAILED:
    print("Skipping post-training evaluation because training failed.")
else:
    after_metrics = trainer.evaluate()

    # Estimate training performance on a small training subset (for generalisation gap)
    TRAIN_EVAL_SUBSET = min(500, len(train_tok))
    train_eval_metrics = trainer.evaluate(eval_dataset=train_tok.select(range(TRAIN_EVAL_SUBSET)))

    comparison = {
        "baseline_accuracy": baseline_metrics.get("eval_accuracy"),
        "after_accuracy": after_metrics.get("eval_accuracy"),
        "baseline_f1_macro": baseline_metrics.get("eval_f1_macro"),
        "after_f1_macro": after_metrics.get("eval_f1_macro"),
        "baseline_loss": baseline_metrics.get("eval_loss"),
        "after_loss": after_metrics.get("eval_loss"),
        "train_subset_accuracy": train_eval_metrics.get("eval_accuracy"),
        "train_subset_f1_macro": train_eval_metrics.get("eval_f1_macro"),
        "train_subset_loss": train_eval_metrics.get("eval_loss"),
    }

    # Generalisation gap (train - val). Positive gap = train looks better than validation.
    comparison["gen_gap_accuracy"] = comparison["train_subset_accuracy"] - comparison["after_accuracy"]
    comparison["gen_gap_f1_macro"] = comparison["train_subset_f1_macro"] - comparison["after_f1_macro"]
    comparison["gen_gap_loss"] = comparison["after_loss"] - comparison["train_subset_loss"]  # loss: higher on val is a gap

    comparison


## 8) Plot the training curve (loss vs epoch)

Use the curve to diagnose behaviour:
- smooth decline -> stable learning
- almost flat -> learning rate too low (for this time budget)
- jumps/explosion -> learning rate too high


In [ ]:
if TRAINING_FAILED:
    print("Skipping curve plot because training failed.")
else:
    log_history = trainer.state.log_history

    # Extract epoch-level logs
    epochs = []
    train_losses = []
    eval_losses = []
    eval_acc = []
    eval_f1 = []

    for row in log_history:
        if "epoch" in row:
            ep = row["epoch"]
            if "loss" in row and "eval_loss" not in row:
                epochs.append(ep)
                train_losses.append(row["loss"])
            if "eval_loss" in row:
                eval_losses.append(row["eval_loss"])
                eval_acc.append(row.get("eval_accuracy"))
                eval_f1.append(row.get("eval_f1_macro"))

    plt.figure()
    if train_losses:
        plt.plot(range(1, len(train_losses)+1), train_losses, marker="o")
    if eval_losses:
        plt.plot(range(1, len(eval_losses)+1), eval_losses, marker="o")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"Loss curves | preset={LR_PRESET} | scheduler={SCHEDULER}")
    plt.legend(["Train loss", "Val loss"])
    plt.tight_layout()
    plt.show()


## 9) Quick qualitative check (before vs after)

This is not a rigorous evaluation, but it helps sanity-check behaviour.


In [ ]:
from transformers import pipeline

# Baseline pipeline always available
baseline_pipe = pipeline("text-classification", model=baseline_model, tokenizer=tokenizer, top_k=None)

trained_pipe = None
if not TRAINING_FAILED:
    trained_pipe = pipeline("text-classification", model=model, tokenizer=tokenizer, top_k=None)

examples = [
    "Shares surged after the company posted record profit.",
    "The firm warned of higher costs due to supply chain disruption.",
    "The company scheduled a call to discuss its dividend policy.",
]

def pretty(preds):
    # preds is list of dicts for each label
    preds_sorted = sorted(preds, key=lambda x: x["score"], reverse=True)
    return ", ".join([f'{p["label"]}:{p["score"]:.2f}' for p in preds_sorted])

print("BASELINE predictions:")
for ex in examples:
    print("-", ex)
    print(" ", pretty(baseline_pipe(ex)[0]))

if trained_pipe is None:
    print("\nTrained predictions skipped (training failed).")
else:
    print("\nAFTER TRAINING predictions:")
    for ex in examples:
        print("-", ex)
        print(" ", pretty(trained_pipe(ex)[0]))


## 10) Reflection (write these in your worksheet)

Answer in plain English, using the correct maths words where helpful:

- What does `trainer.train()` do under the hood?
- What is a **gradient** (in one sentence)?
- What does **stochastic** mean in SGD (in one sentence)?
- What happened to the **generalisation gap** in your run?
- If you had 10× compute, what would you try next (and why)?


## 11) Going further (optional)

Pick one mini‑challenge (time‑boxed):

A) **Schedule comparison:** keep everything the same but set `SCHEDULER = "linear"`.

B) **Freeze encoder:** freeze the base model and train only the classification head.

C) **Error analysis:** confusion matrix + inspect 5 misclassifications.

D) **Experiment tracking (optional):** log your run to W&B and compare two runs (LR presets). See `handouts/Optional_WandB.md`.


In [ ]:
if TRAINING_FAILED:
    print("Skipping confusion matrix because training failed.")
else:
    from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

    # predict on validation set
    pred = trainer.predict(val_tok)
    pred_labels = np.argmax(pred.predictions, axis=-1)
    true_labels = pred.label_ids

    cm = confusion_matrix(true_labels, pred_labels)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
    disp.plot()
    plt.title("Confusion matrix (validation)")
    plt.show()

    # show a few misclassifications (if any)
    val_text = val_ds["text"]
    mis_idx = [i for i, (t, p) in enumerate(zip(true_labels, pred_labels)) if t != p]
    print(f"Misclassifications: {len(mis_idx)}")
    for i in mis_idx[:5]:
        print("---")
        print("Text:", val_text[i])
        print("True:", label_names[true_labels[i]], "| Pred:", label_names[pred_labels[i]])


### Option A — Schedule comparison (guarded)

Set `RUN_SCHEDULE_COMPARISON = True` to run a second short training run using the **same learning rate** but a different schedule.

Tip: keep `EPOCHS = 1` and a small `TRAIN_SUBSET` so this finishes quickly.


In [ ]:
RUN_SCHEDULE_COMPARISON = False

if RUN_SCHEDULE_COMPARISON:
    model2 = build_model()
    args2 = TrainingArguments(
        output_dir="./results_schedule",
        eval_strategy="epoch",
        logging_strategy="epoch",
        save_strategy="no",
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=16,
        learning_rate=learning_rate,
        lr_scheduler_type="linear",
        warmup_ratio=0.1,
        weight_decay=0.01,
        seed=42,
        report_to="none",
    )
    t2 = Trainer(
        model=model2,
        args=args2,
        train_dataset=train_tok,
        eval_dataset=val_tok,
        compute_metrics=compute_metrics,
    )
    t2.train()
    m2 = t2.evaluate()
    print("Schedule comparison metrics (linear):", m2)
else:
    print("Schedule comparison skipped (set RUN_SCHEDULE_COMPARISON=True to run).")


### Option B — Freeze encoder (guarded)

Set `RUN_FREEZE_ENCODER = True` to train only the classification head.

This is a practical trade‑off: sometimes you want cheaper, safer updates even if peak performance is slightly lower.


In [ ]:
RUN_FREEZE_ENCODER = False

def freeze_encoder(m):
    # DistilBERT backbone lives at m.distilbert
    for p in m.distilbert.parameters():
        p.requires_grad = False
    return m

if RUN_FREEZE_ENCODER:
    # Fresh model: this comparison is only meaningful if we start from the same pretrained weights.
    model3 = freeze_encoder(build_model())
    args3 = TrainingArguments(
        output_dir="./results_frozen",
        eval_strategy="epoch",
        logging_strategy="epoch",
        save_strategy="no",
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=16,
        learning_rate=learning_rate,
        lr_scheduler_type=SCHEDULER,
        warmup_ratio=0.1 if SCHEDULER != "constant" else 0.0,
        weight_decay=0.01,
        seed=42,
        report_to="none",
    )
    t3 = Trainer(
        model=model3,
        args=args3,
        train_dataset=train_tok,
        eval_dataset=val_tok,
        compute_metrics=compute_metrics,
    )
    t3.train()
    m3 = t3.evaluate()
    print("Frozen-encoder metrics:", m3)
else:
    print("Freeze encoder skipped (set RUN_FREEZE_ENCODER=True to run).")
